In [ ]:
import pandas as pd

#1. Function used for 1st question
#---------------------------------
def get_production_sessions(production_line_id: str, csv_path: str) -> pd.DataFrame:
    #load the dataset from csv file (delimeter used is ";")
    df = pd.read_csv(csv_path, delimiter=';')
    #convert 'timestamp' column to datetime (so as to be able to proceed with calculations)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    #filter data for prod line and sort by timestamp
    data = df[df['production_line_id'] == production_line_id].sort_values(by='timestamp').reset_index(drop=True)

    sessions = []
    #create 'start_time' variable to track start time of each session
    start_time = None

    #Attention! Take into consideration that termination status is STOP -and not END as mentioned in email's exercise.

    #iterate at filtered rows to find matching START and STOP pairs
    for _, row in data.iterrows():
        if row['status'] == 'START':
            start_time = row['timestamp']           #start of a session
        elif row['status'] == 'STOP' and start_time is not None:
            stop_time = row['timestamp']            #end of a session
            duration = stop_time - start_time       #calculation of duration
            # Append the session details as a dictionary
            sessions.append({
                'start_timestamp': start_time,
                'stop_timestamp': stop_time,
                'duration': str(duration) })        #we convert duration to string so as to be readable
            start_time = None

    #dataframe containing all relevant sessions
    return pd.DataFrame(sessions)

#2. Function used for 2nd question
#---------------------------------
def get_total_uptime_downtime(csv_path: str) -> dict:
    df = pd.read_csv(csv_path, delimiter=';')
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    #sort data chronologically (by production_line_id and timestamp)
    df = df.sort_values(by=['production_line_id', 'timestamp']).reset_index(drop=True)

    #initialization of uptime and downtime
    uptime = pd.Timedelta(0)
    downtime = pd.Timedelta(0)

    #dictionaries to store last status and timestamp per prod line
    last_status = {}
    last_timestamp = {}

    #go through each row and add time between status changes
    for _, row in df.iterrows():
        pl_id = row['production_line_id']
        timestamp = row['timestamp']
        status = row['status']

        #calculation of duration (in case we have met production line before)
        if pl_id in last_status:
            duration = timestamp - last_timestamp[pl_id]
            if last_status[pl_id] == 'ON':
                uptime += duration    #added to uptime if previous status was 'ON'
            else:
                downtime += duration  #added to downtime elsewise

        # update last seen status and timestamp
        last_status[pl_id] = status
        last_timestamp[pl_id] = timestamp

    #return total uptime/downtime (in string format so as to be readable)
    return {'total_uptime': str(uptime), 'total_downtime': str(downtime)}


#3. Function used for 3rd question
#---------------------------------
def get_most_downtime_line(csv_path: str) -> dict:
    df = pd.read_csv(csv_path, delimiter=';')
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values(by=['production_line_id', 'timestamp']).reset_index(drop=True)

    #dictionaries to track downtime per prod line
    downtime_by_line = {}
    last_status = {}
    last_timestamp = {}

    #simalar to (2): go through each row and add downtime for each production line
    for _, row in df.iterrows():
        pl_id = row['production_line_id']
        timestamp = row['timestamp']
        status = row['status']

        if pl_id in last_status:
            duration = timestamp - last_timestamp[pl_id]
            if last_status[pl_id] != 'ON':
                #add to downtime only if previous status was not 'ON'
                if pl_id not in downtime_by_line:
                    downtime_by_line[pl_id] = duration
                else:
                    downtime_by_line[pl_id] += duration

        last_status[pl_id] = status
        last_timestamp[pl_id] = timestamp

    #in case no data for downtime, return empty
    if not downtime_by_line:
        return {}

    #calculation for prod line with the max downtime
    max_line = None
    max_downtime = pd.Timedelta(0)

    for line, downtime in downtime_by_line.items():
      if downtime > max_downtime:
        max_line = line
        max_downtime = downtime

    #return prodlineID and total downtime (in string format so as to be readable)
    return {'production_line_id': max_line, 'downtime': str(max_downtime)}


In [ ]:
#final answers

#1) get production sessions for production line 'gr-np-47'
sessions = get_production_sessions('gr-np-47', 'dataset.csv')
print("Production sessions for 'gr-np-47':")
print(sessions)
print("\n")

#2) get total uptime and downtime for the whole production floor
totals = get_total_uptime_downtime('dataset.csv')
print("Total uptime and downtime for production floor:")
print(totals)
print("\n")

#3) get production line with the most downtime and the total downtime duration of it
most_downtime = get_most_downtime_line('dataset.csv')
print("Production line with most downtime:")
print(most_downtime)


Production sessions for 'gr-np-47':
      start_timestamp      stop_timestamp         duration
0 2020-10-07 01:33:20 2020-10-07 02:03:20  0 days 00:30:00
1 2020-10-07 02:15:02 2020-10-07 04:15:02  0 days 02:00:00
2 2020-10-07 05:00:00 2020-10-07 05:55:17  0 days 00:55:17


Total uptime and downtime for production floor:
{'total_uptime': '0 days 15:22:39', 'total_downtime': '0 days 01:34:18'}


Production line with most downtime:
{'production_line_id': 'gr-np-47', 'downtime': '0 days 01:24:18'}
